In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from tqdm import tqdm
import colorir as cl
from analyses.rust import contact
from pathlib import Path

In [ ]:
latpaths = []
for energy in Path("../runs/ji/").iterdir():
    for act in energy.iterdir():
        latpaths.append(act / "lattices")
latpaths

In [ ]:
latdf = pl.read_parquet(latpaths, include_file_paths="path")
latdf

In [ ]:
neigh_maps = {}
for (path,), clat in tqdm(latdf.group_by("path"), total=latdf["path"].n_unique()):
    neigh_maps[path] = contact.neighbour_map(clat.select(pl.exclude("path")), False)
neigh_maps

In [ ]:
pathdf = pl.DataFrame(list(neigh_maps.keys())).with_columns(list=pl.col("column_0").str.split("/")).with_columns(
    wtime=pl.col("list").list.last().str.replace(".parquet", "").cast(int),
    act=pl.col("list").list.get(-3).str.replace("act-", "").cast(int),
    gamma=20 - pl.col("list").list.get(-4).str.replace("energy-", "").cast(int),
).sort(["gamma", "act", "wtime"])
pathdf

In [ ]:
def jaccards_index(set1: set, set2):
    union = len(set1.union(set2))
    if union == 0:
        return None
    inter = len(set1.intersection(set2))
    return inter / union

jis = []
rows = list(pathdf.iter_rows())
for row1, row2 in tqdm(zip(rows, rows[1:]), total=len(rows) - 1):
    if row1[-1] != row2[-1] or row1[-2] != row2[-2]:
        continue

    index1 = row1[2:]
    index2 = row2[2:]
    newindex = (index1[0], index2[0], *index1[1:])
    
    neighmap1 = neigh_maps[row1[0]]
    neighmap2 = neigh_maps[row2[0]]

    for spin, neighs1 in neighmap1.items():
        discard = {"s", "m"}  # We don't necessarily need to discard these but its prob a good idea
        ji = jaccards_index(neighs1 - discard, neighmap2[spin] - discard)
        jis.append((*newindex, spin, ji))
jidf = pl.from_records(jis, orient="row", schema=["time1", "time2", "act", "gamma", "spin", "ji"])
jidf = jidf.with_columns(jd=1 - pl.col("ji"))
jidf

In [ ]:
aggdf = (
    jidf
        .filter(~pl.col("spin").is_in(["s", "m"]))
        .group_by(["gamma", "act"])
        .agg(
            mean=pl.col("jd").mean(),
            std=pl.col("jd").std(),
            med=pl.col("jd").median(),
            min=pl.col("jd").min(),
            max=pl.col("jd").max(),
        )
).sort(["gamma", "act"])
aggdf

In [ ]:
actdf = aggdf.filter(act=160)
fig = go.Figure()
fig.add_traces([
    go.Scatter(
        x=pl.concat([actdf["gamma"], actdf["gamma"][::-1]]),
        y=pl.concat([actdf["mean"] + actdf["std"], (actdf["mean"] - actdf["std"])[::-1]]),
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.2)",
        line_color="rgba(0, 0, 0, 0)"
    ),
    go.Scatter(
        x=actdf["gamma"],
        y=actdf["mean"],
        line_color="#000000"
    ),
])
fig.update_layout(
    template="plotly_white",
    width=500,
    height=500
    #showlegend=False
)